# 多项式回归示例

## 目标

理解多项式回归的工作原理和应用场景。

- 理解为什么需要多项式回归
- 从零实现多项式回归
- 对比线性回归和多项式回归
- 理解过拟合问题及其解决方法

## 1. 数据准备 - 非线性关系

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import pandas as pd

np.random.seed(42)

# 生成非线性数据
n_samples = 100
X = np.linspace(-3, 3, n_samples).reshape(-1, 1)

# 真实的二次关系 y = 2x^2 + x - 3 + noise
true_y = 2 * X.ravel()**2 + X.ravel() - 3
y = true_y + np.random.randn(n_samples) * 2

print(f"数据形状: X={X.shape}, y={y.shape}")
print(f"X范围: [{X.min():.2f}, {X.max():.2f}]")
print(f"y范围: [{y.min():.2f}, {y.max():.2f}]")

## 2. 可视化数据和真实关系

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.6, label='观测数据')
plt.plot(X, true_y, 'r-', linewidth=2, label='真实关系')
plt.xlabel('X')
plt.ylabel('y')
plt.title('非线性数据分布')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. 线性回归的局限性

In [ ]:
# 使用线性回归
linear_model = LinearRegression()
linear_model.fit(X, y)

y_pred_linear = linear_model.predict(X)

print("线性回归结果:")
print(f"斜率: {linear_model.coef_[0]:.4f}")
print(f"截距: {linear_model.intercept_:.4f}")
print(f"R²得分: {r2_score(y, y_pred_linear):.4f}")
print(f"MSE: {mean_squared_error(y, y_pred_linear):.4f}")

In [ ]:
# 可视化线性回归结果
plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.6, label='观测数据')
plt.plot(X, true_y, 'r-', linewidth=2, label='真实关系')
plt.plot(X, y_pred_linear, 'g--', linewidth=2, label='线性回归拟合')
plt.xlabel('X')
plt.ylabel('y')
plt.title('线性回归 vs 真实关系')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. 多项式回归 - 从零实现

In [ ]:
def polynomial_features_manual(X, degree):
    """
    手动实现多项式特征变换
    
    Parameters:
    -----------
    X : array-like, shape (n_samples, 1)
        输入特征
    degree : int
        多项式次数
    
    Returns:
    --------
    X_poly : array, shape (n_samples, degree+1)
        多项式特征矩阵 [1, x, x², ..., x^degree]
    """
    n_samples = X.shape[0]
    X_poly = np.ones((n_samples, degree + 1))
    
    for i in range(1, degree + 1):
        X_poly[:, i] = X.ravel() ** i
    
    return X_poly

def fit_polynomial_regression(X, y, degree):
    """
    拟合多项式回归模型
    
    Parameters:
    -----------
    X : array-like, shape (n_samples, 1)
        输入特征
    y : array-like, shape (n_samples,)
        目标值
    degree : int
        多项式次数
    
    Returns:
    --------
    model : LinearRegression
        训练好的线性回归模型
    """
    # 特征变换
    X_poly = polynomial_features_manual(X, degree)
    
    # 拟合线性回归
    model = LinearRegression()
    model.fit(X_poly, y)
    
    return model

print("多项式回归工具函数定义完成")

## 5. 多项式回归 - 二次回归

In [ ]:
# 二次多项式回归
degree_2 = 2
poly_model_2 = fit_polynomial_regression(X, y, degree_2)

X_poly_2 = polynomial_features_manual(X, degree_2)
y_pred_poly_2 = poly_model_2.predict(X_poly_2)

print(f"二次多项式回归结果:")
print(f"系数: {poly_model_2.coef_.round(4)}")
print(f"截距: {poly_model_2.intercept_:.4f}")
print(f"R²得分: {r2_score(y, y_pred_poly_2):.4f}")
print(f"MSE: {mean_squared_error(y, y_pred_poly_2):.4f}")

print(f"\n拟合的方程: y = {poly_model_2.coef_[2]:.4f}x² + {poly_model_2.coef_[1]:.4f}x + {poly_model_2.coef_[0]:.4f}")

In [ ]:
# 可视化二次多项式回归结果
plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.6, label='观测数据')
plt.plot(X, true_y, 'r-', linewidth=2, label='真实关系')
plt.plot(X, y_pred_linear, 'g--', linewidth=2, label='线性回归')
plt.plot(X, y_pred_poly_2, 'b-', linewidth=2, label='二次多项式回归')
plt.xlabel('X')
plt.ylabel('y')
plt.title('不同回归模型对比')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. 使用 Scikit-learn 的 PolynomialFeatures

In [ ]:
# 使用 Scikit-learn 的多项式特征
poly_features = PolynomialFeatures(degree=2, include_bias=False)
X_poly_sklearn = poly_features.fit_transform(X)

print("多项式特征形状:")
print(f"原始X: {X.shape}")
print(f"变换后X: {X_poly_sklearn.shape}")

print("\n前5个样本的特征:")
for i in range(5):
    print(f"x={X[i,0]:6.3f} → [x, x²] = {X_poly_sklearn[i]}")

In [ ]:
# 使用 Pipeline
poly_pipeline = Pipeline([
    ('poly_features', PolynomialFeatures(degree=2, include_bias=True)),
    ('linear_regression', LinearRegression())
])

poly_pipeline.fit(X, y)
y_pred_pipeline = poly_pipeline.predict(X)

print("使用 Pipeline 的结果:")
print(f"R²得分: {r2_score(y, y_pred_pipeline):.4f}")
print(f"系数: {poly_pipeline.named_steps['linear_regression'].coef_.round(4)}")
print(f"截距: {poly_pipeline.named_steps['linear_regression'].intercept_:.4f}")

## 7. 不同次数的多项式回归对比

In [ ]:
# 测试不同次数的多项式
degrees = [1, 2, 3, 5, 10, 15]
colors = ['green', 'blue', 'orange', 'purple', 'brown', 'pink']

plt.figure(figsize=(15, 5))

# 第一行：低次多项式
plt.subplot(1, 2, 1)
plt.scatter(X, y, alpha=0.4, label='观测数据', color='gray')
plt.plot(X, true_y, 'k-', linewidth=2, label='真实关系')

for degree, color in zip(degrees[:3], colors[:3]):
    poly_model = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=True)),
        ('linear', LinearRegression())
    ])
    poly_model.fit(X, y)
    y_pred = poly_model.predict(X)
    r2 = r2_score(y, y_pred)
    plt.plot(X, y_pred, linewidth=2, label=f'degree={degree}, R²={r2:.3f}', color=color)

plt.xlabel('X')
plt.ylabel('y')
plt.title('低次多项式回归')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(-5, 25)

# 第二行：高次多项式
plt.subplot(1, 2, 2)
plt.scatter(X, y, alpha=0.4, label='观测数据', color='gray')
plt.plot(X, true_y, 'k-', linewidth=2, label='真实关系')

for degree, color in zip(degrees[3:], colors[3:]):
    poly_model = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=True)),
        ('linear', LinearRegression())
    ])
    poly_model.fit(X, y)
    y_pred = poly_model.predict(X)
    r2 = r2_score(y, y_pred)
    plt.plot(X, y_pred, linewidth=2, label=f'degree={degree}, R²={r2:.3f}', color=color)

plt.xlabel('X')
plt.ylabel('y')
plt.title('高次多项式回归')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(-5, 25)

plt.tight_layout()
plt.show()

## 8. 过拟合问题

In [ ]:
# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"训练集大小: {len(X_train)}")
print(f"测试集大小: {len(X_test)}")

In [ ]:
# 测试不同次数在训练集和测试集上的表现
degrees = list(range(1, 16))

train_scores = []
test_scores = []
train_mse = []
test_mse = []

for degree in degrees:
    poly_model = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=True)),
        ('linear', LinearRegression())
    ])
    
    poly_model.fit(X_train, y_train)
    
    y_train_pred = poly_model.predict(X_train)
    y_test_pred = poly_model.predict(X_test)
    
    train_scores.append(r2_score(y_train, y_train_pred))
    test_scores.append(r2_score(y_test, y_test_pred))
    train_mse.append(mean_squared_error(y_train, y_train_pred))
    test_mse.append(mean_squared_error(y_test, y_test_pred))

# 找到最佳次数
best_degree = np.argmax(test_scores) + 1
print(f"最佳多项式次数: {best_degree}")
print(f"最佳测试集 R²: {test_scores[best_degree-1]:.4f}")

In [ ]:
# 可视化过拟合
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# R²得分
axes[0].plot(degrees, train_scores, 'b-o', label='训练集 R²', linewidth=2, markersize=6)
axes[0].plot(degrees, test_scores, 'r-o', label='测试集 R²', linewidth=2, markersize=6)
axes[0].axvline(best_degree, color='gray', linestyle='--', label=f'最佳次数={best_degree}')
axes[0].set_xlabel('多项式次数')
axes[0].set_ylabel('R² 得分')
axes[0].set_title('训练集 vs 测试集 R² 得分')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-2, 1.1)

# MSE
axes[1].plot(degrees, train_mse, 'b-o', label='训练集 MSE', linewidth=2, markersize=6)
axes[1].plot(degrees, test_mse, 'r-o', label='测试集 MSE', linewidth=2, markersize=6)
axes[1].axvline(best_degree, color='gray', linestyle='--', label=f'最佳次数={best_degree}')
axes[1].set_xlabel('多项式次数')
axes[1].set_ylabel('MSE')
axes[1].set_title('训练集 vs 测试集 MSE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

## 9. 可视化不同次数的拟合效果

In [ ]:
# 可视化不同次数的拟合效果
selected_degrees = [1, 2, 3, best_degree, 10]

plt.figure(figsize=(15, 8))

# 使用更密集的点绘制平滑曲线
X_smooth = np.linspace(-3, 3, 200).reshape(-1, 1)

for i, degree in enumerate(selected_degrees):
    plt.subplot(2, 3, i+1)
    
    poly_model = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=True)),
        ('linear', LinearRegression())
    ])
    
    poly_model.fit(X_train, y_train)
    
    y_train_pred = poly_model.predict(X_train)
    y_test_pred = poly_model.predict(X_test)
    y_smooth_pred = poly_model.predict(X_smooth)
    
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    plt.scatter(X_train, y_train, alpha=0.4, color='blue', label='训练集')
    plt.scatter(X_test, y_test, alpha=0.4, color='red', label='测试集')
    plt.plot(X_smooth, y_smooth_pred, 'g-', linewidth=2, label='拟合曲线')
    
    plt.xlabel('X')
    plt.ylabel('y')
    plt.title(f'次数={degree}\n训练R²={train_r2:.3f}, 测试R²={test_r2:.3f}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.ylim(-5, 25)

plt.tight_layout()
plt.show()

## 10. 处理过拟合的方法

In [ ]:
# 生成更复杂的数据
np.random.seed(42)
n_samples = 30  # 小样本，更容易过拟合
X_complex = np.linspace(-2, 2, n_samples).reshape(-1, 1)
y_complex = np.sin(3 * X_complex).ravel() + np.random.randn(n_samples) * 0.2

# 划分数据
X_train_complex, X_test_complex, y_train_complex, y_test_complex = train_test_split(
    X_complex, y_complex, test_size=0.3, random_state=42
)

print(f"复杂数据集: 训练集={len(X_train_complex)}, 测试集={len(X_test_complex)}")

In [ ]:
# 比较正则化和非正则化
from sklearn.linear_model import Ridge

degree = 15

# 无正则化
poly_no_reg = Pipeline([
    ('poly', PolynomialFeatures(degree=degree, include_bias=True)),
    ('linear', LinearRegression())
])
poly_no_reg.fit(X_train_complex, y_train_complex)

# 有正则化 (Ridge)
poly_ridge = Pipeline([
    ('poly', PolynomialFeatures(degree=degree, include_bias=True)),
    ('ridge', Ridge(alpha=0.1))
])
poly_ridge.fit(X_train_complex, y_train_complex)

# 计算指标
y_train_pred_no_reg = poly_no_reg.predict(X_train_complex)
y_test_pred_no_reg = poly_no_reg.predict(X_test_complex)
y_train_pred_ridge = poly_ridge.predict(X_train_complex)
y_test_pred_ridge = poly_ridge.predict(X_test_complex)

print("无正则化:")
print(f"  训练集 R²: {r2_score(y_train_complex, y_train_pred_no_reg):.4f}")
print(f"  测试集 R²: {r2_score(y_test_complex, y_test_pred_no_reg):.4f}")

print("\nRidge 正则化:")
print(f"  训练集 R²: {r2_score(y_train_complex, y_train_pred_ridge):.4f}")
print(f"  测试集 R²: {r2_score(y_test_complex, y_test_pred_ridge):.4f}")

In [ ]:
# 可视化正则化效果
X_smooth_complex = np.linspace(-2.5, 2.5, 200).reshape(-1, 1)

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.scatter(X_train_complex, y_train_complex, alpha=0.6, color='blue', label='训练集')
plt.scatter(X_test_complex, y_test_complex, alpha=0.6, color='red', label='测试集')
plt.plot(X_smooth_complex, poly_no_reg.predict(X_smooth_complex), 'g-', linewidth=2, label='拟合曲线')
plt.xlabel('X')
plt.ylabel('y')
plt.title(f'无正则化 (过拟合)\n训练R²={r2_score(y_train_complex, y_train_pred_no_reg):.3f}, 测试R²={r2_score(y_test_complex, y_test_pred_no_reg):.3f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(-1.5, 1.5)

plt.subplot(1, 2, 2)
plt.scatter(X_train_complex, y_train_complex, alpha=0.6, color='blue', label='训练集')
plt.scatter(X_test_complex, y_test_complex, alpha=0.6, color='red', label='测试集')
plt.plot(X_smooth_complex, poly_ridge.predict(X_smooth_complex), 'g-', linewidth=2, label='拟合曲线')
plt.xlabel('X')
plt.ylabel('y')
plt.title(f'Ridge 正则化 (平滑)\n训练R²={r2_score(y_train_complex, y_train_pred_ridge):.3f}, 测试R²={r2_score(y_test_complex, y_test_pred_ridge):.3f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(-1.5, 1.5)

plt.tight_layout()
plt.show()

## 11. 不同正则化强度的对比

In [ ]:
# 测试不同正则化强度
alphas = [0, 0.001, 0.01, 0.1, 1, 10]

plt.figure(figsize=(15, 8))

for i, alpha in enumerate(alphas):
    plt.subplot(2, 3, i+1)
    
    if alpha == 0:
        poly_temp = Pipeline([
            ('poly', PolynomialFeatures(degree=15, include_bias=True)),
            ('linear', LinearRegression())
        ])
    else:
        poly_temp = Pipeline([
            ('poly', PolynomialFeatures(degree=15, include_bias=True)),
            ('ridge', Ridge(alpha=alpha))
        ])
    
    poly_temp.fit(X_train_complex, y_train_complex)
    
    y_train_pred_temp = poly_temp.predict(X_train_complex)
    y_test_pred_temp = poly_temp.predict(X_test_complex)
    
    plt.scatter(X_train_complex, y_train_complex, alpha=0.4, color='blue')
    plt.scatter(X_test_complex, y_test_complex, alpha=0.4, color='red')
    plt.plot(X_smooth_complex, poly_temp.predict(X_smooth_complex), 'g-', linewidth=2)
    
    plt.title(f'α={alpha}\n训练R²={r2_score(y_train_complex, y_train_pred_temp):.3f}, 测试R²={r2_score(y_test_complex, y_test_pred_temp):.3f}')
    plt.grid(True, alpha=0.3)
    plt.ylim(-1.5, 1.5)

plt.tight_layout()
plt.show()

## 12. 实践建议总结

In [ ]:
# 创建多项式回归的最佳实践函数
def polynomial_regression_with_cv(X, y, max_degree=10, cv_folds=5):
    """
    使用交叉验证选择最佳多项式次数
    
    Parameters:
    -----------
    X : array-like, shape (n_samples, 1)
        输入特征
    y : array-like, shape (n_samples,)
        目标值
    max_degree : int
        最大多项式次数
    cv_folds : int
        交叉验证折数
    
    Returns:
    --------
    best_model : Pipeline
        最佳模型
    best_degree : int
        最佳多项式次数
    results : dict
        详细结果
    """
    from sklearn.model_selection import cross_val_score
    
    cv_scores = []
    models = []
    
    for degree in range(1, max_degree + 1):
        poly_model = Pipeline([
            ('poly', PolynomialFeatures(degree=degree, include_bias=True)),
            ('linear', LinearRegression())
        ])
        
        scores = cross_val_score(poly_model, X, y, cv=cv_folds, scoring='r2')
        cv_scores.append(scores.mean())
        models.append(poly_model)
    
    best_degree = np.argmax(cv_scores) + 1
    best_model = models[best_degree - 1]
    best_model.fit(X, y)
    
    results = {
        'degrees': list(range(1, max_degree + 1)),
        'cv_scores': cv_scores,
        'best_degree': best_degree,
        'best_cv_score': cv_scores[best_degree - 1]
    }
    
    return best_model, best_degree, results

# 使用示例
best_model, best_degree, results = polynomial_regression_with_cv(X, y, max_degree=10, cv_folds=5)
print(f"交叉验证选择的最佳多项式次数: {best_degree}")
print(f"最佳交叉验证得分: {results['best_cv_score']:.4f}")

## 总结

### 多项式回归 vs 线性回归

| 特性 | 线性回归 | 多项式回归 |
|------|----------|-----------|
| 模型形式 | y = β₀ + β₁x | y = β₀ + β₁x + β₂x² + ... + βₙxⁿ |
| 拟合能力 | 只能拟合直线 | 可拟合曲线 |
| 参数数量 | 2 | degree+1 |
| 过拟合风险 | 低 | 高（特别是高次） |

### 关键要点

1. **何时使用多项式回归**:
   - 数据明显呈非线性关系
   - 可视化显示曲线趋势
   - 残差图显示模式

2. **选择多项式次数**:
   - 从低次开始（2、3次）
   - 使用交叉验证
   - 观察训练/测试集性能差异

3. **防止过拟合**:
   - 使用正则化 (Ridge/Lasso)
   - 增加训练数据
   - 选择合适的多项式次数
   - 使用交叉验证

4. **实现要点**:
   - 使用 PolynomialFeatures 进行特征变换
   - 必须标准化特征（特别是在高次时）
   - 使用 Pipeline 保持代码整洁

### 实践建议

- 默认从 2 次多项式开始尝试
- 优先使用交叉验证选择次数
- 高次多项式时务必使用正则化
   - Ridge: 系数收缩，平滑拟合
   - Lasso: 可能产生稀疏解
- 始终可视化拟合结果
- 比较训练集和测试集性能以检测过拟合